# Evaluation Sampling + REsearcher Annotation + Fleiss' Kappa

In [1]:
import os                                                                                                 
import numpy as np                                                                                        
import pandas as pd                                                                                       
from sklearn.model_selection import train_test_split                                                      
from statsmodels.stats.inter_rater import fleiss_kappa                                                    
                                                                                                            
# Enforce strict reproducibility                                                                          
SEED = 42                                                                                                 
np.random.seed(SEED)


In [2]:
# =============================================================================                           
# Sub-step 4.2.1: Evaluation Sampling                                                                     
# Jointly stratify by TargetAspect and proxy star rating buckets:                                         
# - 1 to 2 stars: Negative proxy                                                                          
# - 3 stars: Neutral proxy                                                                                
# - 4 to 5 stars: Positive proxy                                                                          
# Sample 400 from Validation and 400 from Test (800 instances total).                                     
# =============================================================================                           
                                                                                                            
def assign_rating_bucket(rating):                                                                         
    r = int(rating)                                                                                       
    if r <= 2:                                                                                            
        return '1-2_stars'                                                                                
    elif r == 3:                                                                                          
        return '3_stars'                                                                                  
    return '4-5_stars'                                                                                    
                                                                                                            
def draw_stratified_evaluation_sample(input_csv, sample_size=400, seed=SEED):                             
    df = pd.read_csv(input_csv)                                                                           
                                                                                                            
    # Create joint stratification key                                                                     
    df['rating_bucket'] = df['rating'].apply(assign_rating_bucket)                                        
    df['stratum'] = df['TargetAspect'].astype(str) + " | " + df['rating_bucket']                          
                                                                                                            
    # Exact stratified draw                                                                               
    sample_df, _ = train_test_split(                                                                      
        df,                                                                                               
        train_size=sample_size,                                                                           
        stratify=df['stratum'],                                                                           
        random_state=seed,                                                                                
        shuffle=True                                                                                      
    )                                                                                                     
                                                                                                            
    # Prepare annotation columns for 3 independent raters                                                 
    base_cols = ['place_name', 'rating', 'shop_type', 'sentence', 'TargetAspect']                         
    export_df = sample_df[base_cols].copy().reset_index(drop=True)                                        
    export_df['annotator_1'] = np.nan                                                                     
    export_df['annotator_2'] = np.nan                                                                     
    export_df['annotator_3'] = np.nan                                                                     
                                                                                                            
    return export_df                                                                                      
                                                                                                            
# Draw samples                                                                                            
val_gold_unannotated = draw_stratified_evaluation_sample('Validation_Set_15.csv', sample_size=400)        
test_gold_unannotated = draw_stratified_evaluation_sample('Test_Set_15.csv', sample_size=400)             
                                                                                                            
# Save unannotated templates for the 3 human raters                                                       
val_gold_unannotated.to_csv('Gold_Validation_Set_400_Unannotated.csv', index=False)                       
test_gold_unannotated.to_csv('Gold_Test_Set_400_Unannotated.csv', index=False)                            
                                                                                                            
print(f"Generated Gold_Validation_Set_400_Unannotated.csv: {val_gold_unannotated.shape}")                 
print(f"Generated Gold_Test_Set_400_Unannotated.csv: {test_gold_unannotated.shape}")

Generated Gold_Validation_Set_400_Unannotated.csv: (400, 8)
Generated Gold_Test_Set_400_Unannotated.csv: (400, 8)


In [3]:
# Verify distribution across aspects and proxy buckets                                                    
def display_sampling_strata(df, name):                                                                    
    print(f"\n=== {name} Stratified Distribution (n={len(df)}) ===")                                      
    df_temp = df.copy()                                                                                   
    df_temp['rating_bucket'] = df_temp['rating'].apply(assign_rating_bucket)                              
    ct = pd.crosstab(df_temp['TargetAspect'], df_temp['rating_bucket'], margins=True,                     
margins_name='Total')                                                                                       
    print(ct.to_markdown())                                                                               
                                                                                                            
display_sampling_strata(val_gold_unannotated, "Validation Sample")                                        
display_sampling_strata(test_gold_unannotated, "Test Sample")


=== Validation Sample Stratified Distribution (n=400) ===
| TargetAspect                       |   1-2_stars |   3_stars |   4-5_stars |   Total |
|:-----------------------------------|------------:|----------:|------------:|--------:|
| Ambiance and Atmosphere            |           3 |         3 |          47 |      53 |
| Customer Service                   |          15 |         6 |          59 |      80 |
| Facilities and Amenities           |           4 |         3 |          26 |      33 |
| Price and Value                    |           3 |         3 |          15 |      21 |
| Product Quality                    |          21 |        14 |         157 |     192 |
| Store Operations and Accessibility |           3 |         2 |          16 |      21 |
| Total                              |          49 |        31 |         320 |     400 |

=== Test Sample Stratified Distribution (n=400) ===
| TargetAspect                       |   1-2_stars |   3_stars |   4-5_stars |   Total 